# Grant App Service Principal Unity Catalog Permissions

This notebook grants the Databricks App's service principal access to Unity Catalog resources.

**Purpose:** When a Databricks App is deployed, it runs under its own service principal. This principal needs explicit permissions to query tables in Unity Catalog.

**Permissions Granted:**
- `USE CATALOG` on the target catalog
- `USE SCHEMA` on the target schema  
- `SELECT` on all tables in the schema

**Prerequisites:**
- The Databricks App must already be deployed
- You must have permission to grant UC privileges

## Configuration

These parameters are passed from the job definition. When running interactively, you can set them manually.

In [ ]:
from __future__ import annotations

import os

def _get_config(name: str, default: str) -> str:
    """Get config from job parameters (widgets), env vars, or default."""
    try:
        return dbutils.widgets.get(name)  # type: ignore[name-defined]
    except Exception:
        return os.environ.get(f"DATABRICKS_{name.upper()}", default)

# App name - must match the deployed app
APP_NAME = _get_config("app_name", "logistics-incident-response")

# SQL Warehouse for executing GRANT statements
WAREHOUSE_ID = _get_config("warehouse_id", "106bb1811885f66e")

# Unity Catalog location
CATALOG = _get_config("catalog", "main")
SCHEMA = _get_config("schema", "logistics_control_center")

print(f"Configuration:")
print(f"  App Name: {APP_NAME}")
print(f"  Warehouse ID: {WAREHOUSE_ID}")
print(f"  Catalog: {CATALOG}")
print(f"  Schema: {SCHEMA}")

## Initialize Databricks SDK Client

The WorkspaceClient automatically uses the notebook's authentication context when running on Databricks.

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import StatementState

client = WorkspaceClient()
print(f"Connected to workspace: {client.config.host}")

## Retrieve App Service Principal

Each Databricks App has an associated service principal. We need to retrieve its ID to grant permissions.

The service principal ID is typically a UUID (e.g., `5e30eb11-8c94-4fee-9714-01d05cb0fdf9`).

In [ ]:
app = client.api_client.do("GET", f"/api/2.0/apps/{APP_NAME}")

principal = app.get("service_principal_client_id") or app.get("service_principal_name")

if not principal:
    raise RuntimeError(
        f"App '{APP_NAME}' does not expose a service principal identifier. "
        "Ensure the app is deployed and has a service principal assigned."
    )

print(f"App: {APP_NAME}")
print(f"Service Principal ID: {principal}")

## Helper Function: Execute SQL Statement

This function executes a SQL statement using the Databricks SQL Statement Execution API.

We use this instead of `spark.sql()` because GRANT statements work more reliably through the Statement Execution API.

In [ ]:
def _quote_principal(principal: str) -> str:
    """Escape backticks in principal name for SQL."""
    return f"`{principal.replace('`', '``')}`"


def run_sql(statement: str) -> None:
    """Execute a SQL statement via the Statement Execution API."""
    print(f"Executing: {statement}")
    
    execution = client.statement_execution.execute_statement(
        warehouse_id=WAREHOUSE_ID,
        statement=statement,
        wait_timeout="40s",
    )
    
    state = execution.status.state if execution.status else None
    if state != StatementState.SUCCEEDED:
        error = execution.status.error if execution.status else "unknown"
        raise RuntimeError(f"SQL failed with state={state}: {error}")
    
    print(f"  ✓ Success")

## Grant Unity Catalog Permissions

We grant three levels of permissions:

1. **USE CATALOG** - Allows the principal to see the catalog exists
2. **USE SCHEMA** - Allows the principal to see the schema exists  
3. **SELECT ON SCHEMA** - Allows the principal to read all tables in the schema

These permissions enable the app to query the logistics data tables.

In [ ]:
fq_schema = f"{CATALOG}.{SCHEMA}"
principal_sql = _quote_principal(principal)

statements = [
    f"GRANT USE CATALOG ON CATALOG {CATALOG} TO {principal_sql}",
    f"GRANT USE SCHEMA ON SCHEMA {fq_schema} TO {principal_sql}",
    f"GRANT SELECT ON SCHEMA {fq_schema} TO {principal_sql}",
]

print(f"Granting permissions to service principal: {principal}")
print(f"Target: {fq_schema}")
print()

for stmt in statements:
    run_sql(stmt)

print()
print(f"✓ All permissions granted successfully!")
print(f"  App '{APP_NAME}' can now query tables in {fq_schema}")

## Verification (Optional)

You can verify the grants were applied by checking the principal's permissions.

In [ ]:
print(f"To verify permissions, run:")
print(f"  SHOW GRANTS TO {principal_sql}")
print()
print(f"App URL: https://{client.config.host.replace('https://', '').replace('.azuredatabricks.net', '')}.azuredatabricksapps.com/{APP_NAME}")